# 第三轮讨论：架构设计深入

## 讨论背景

前两轮讨论已收敛的共识：
- 核心方法：Per-finger RMA + Graph Attention
- 应用靶点：手型 × 物体双向泛化的 in-hand rotation
- Token 粒度：joint-level（可与 per-finger adaptation 解耦）
- Hand-side 机制：dynamic state × static embodiment 的 cross-attention
- Object-side：per-finger local latent → aggregation → global memory
- Local 主路 + Global 残差

本轮聚焦：**架构设计的具体细节**

## 3.1 架构拍板顺序：先定 memory 形态，再定交互链路

**用户反馈**：前两轮已接受 `joint-level token + per-finger local latent + global memory + hand-side cross-attention` 的总体骨架；本轮希望聚焦能直接决定编码实现的架构细节，并重点考虑 20Hz+ 推理约束与 joint-space action 硬约束。

**分析**：

当前 5 个悬留问题并不是彼此独立的。若按“最少返工”的角度看，建议按下面顺序拍板：

| 优先级 | 要先决定的点 | 原因 | 对后续影响 |
|---|---|---|---|
| 1 | `object memory` 是 1 个 token 还是多个 memory tokens | 这直接决定 cross-attention 到底只是全局门控，还是能做多接触区域的信息路由 | 会影响 supervision、finger summary 是否必要、以及时序编码应输出给谁 |
| 2 | 时序 history 放图外还是图内 | 20Hz+ 约束下，这决定整网复杂度上限 | 会影响 local latent 生成方式和 hand/object 交互带宽 |
| 3 | local/global 的监督拆分 | 决定 per-finger latent 是否真的学到“局部交互”，还是退化成重复版全局 latent | 会影响实验可解释性 |
| 4 | 是否显式加 finger summary layer | 主要影响未来多手型扩展与结构可解释性 | 对单手型 MVP 不是第一优先 |
| 5 | cross-attention 具体放哪几层 | 本质上依赖前面几项，因为 memory/token 形态不同，最优交互链路也会变 | 是“后置拍板项” |

从目前约束看，我更倾向把设计空间先压缩成两类 object-side 方案：

1. **单 memory token（HORA-plus 版本）**
   - 形式：`{z_f} -> aggregate -> z_g`，再由 joint tokens 读取 `z_g`
   - 优点：最快、最稳、实现最简；更像“全局协调残差”
   - 风险：cross-attention 容易退化成高级版 gating，难体现“多局部接触路由”的结构优势

2. **少量 memory tokens / slots（推荐作为更有 paper 味的候选）**
   - 形式：`{z_f} -> K 个 object memory slots`，joint/finger summary 再去 cross-attend
   - 优点：能表达“多个局部接触假设 / 多区域交互摘要”，更匹配不对称物体与接触角色切换
   - 风险：如果 K 太大就会逼近重型 object graph；需要控制到很小（如 2~4 个）才能守住 20Hz+

若以 **第一版可编码 + 20Hz+ + 仍保留结构新意** 为目标，一个很像样的折中是：
- **history encoder 放图外**：每指一个小 TCN/GRU，输出 `z_f`
- **object side 用少量固定上限 slots（如 2 或 4）**，而不是单一 `z_g`
- **hand 内部仍用 self-attention + graph bias**
- **hand -> object / object -> hand 才用 cross-attention**

这会让整套东西更像：
`finger-local temporal inference -> compact object memory -> embodiment-aware hand graph -> per-joint action`

而不是把所有问题都丢给一个大 transformer。

**小结**：如果本轮要收敛到“可以开始编码”的层面，第一拍最好先落在 **object memory 的形态与数量上**；这会把后面 4 个问题一起缩到更小的搜索空间。

**待确认**：第一版 MVP，你更倾向 `1 个全局 memory token`，还是 `少量 memory slots（2~4）`？

## 3.2 补读必读论文与代码后的结构约束

**用户反馈**：在继续拍板 object memory 之前，先补读 `AnyMani/source/anymani/papers/context.md` 中列出的必读论文与代码，以免第三轮架构讨论脱离证据。

**分析**：

本轮已补读：
- 论文：HORA 2022、RotateIt 2023、MAGCLA 2023、AnyRotate 2024、GET-Zero 2024、DexNDM 2025、T(R,O) Grasp 2025
- 代码：`TRO-Grasp/model/tro_graph.py`、`TRO-Grasp/model/denoiser.py`、`get_zero/get_zero/rl/models/embodiment_transformer.py`、`get_zero/get_zero/distill/models/embodiment_attention.py`、`get_zero/get_zero/distill/models/embodiment_transformer.py`、`get_zero/get_zero/distill/models/vis_embodiment_transformer.py`

补读后的结论不是“推翻前两轮”，而是把第三轮的搜索空间进一步压缩了：

### 一、关于时序处理：**图外 history encoder 更像默认优选**

1. **HORA** 的 adaptation 模块明确是图外时序编码：先用 proprio/action history 预测 extrinsics，再喂给控制策略。
2. **AnyRotate** 的 student 也是 `history -> latent -> policy` 结构，而不是把长时间序列扔进整个图网络里。
3. **HORA 附录** 还给了一个很关键的负面证据：
   - DR baseline 里直接 concat 长 history 的 MLP 很难训；
   - LSTM 也不理想；
   - TCN 风格更稳。
4. **RotateIt** 虽然用了 transformer 做时序融合，但它面对的是 vision+touch+proprio 多模态序列，而不是我们当前更轻的 proprio/contact 主设定。

**结构含义**：如果我们目标是 `20Hz+` 且第一版仍以 proprio / light tactile 为主，那么更像样的默认方案是：
- 每根手指单独一个小 TCN/GRU 编码 history，输出 `z_f`
- hand graph 只处理“当前时刻的 joint tokens + object memory”
- 不做时空一锅炖的大 transformer

### 二、关于 hand-side：**GET-Zero 的真正贡献点依然是 joint token + graph bias**

从论文和源码都能看清：
- token 单位是 **joint / DoF**，不是 finger；
- 图归纳偏置是加在 attention score 上的 **SPD + parent/child + edge bias**；
- 它在 hand-generalization 上最关键的是：
  1. token 粒度足够细；
  2. embodiment 信息不是简单 concat，而是通过图结构/偏置参与注意力。

**结构含义**：
- 我们把 token 粒度定在 joint-level 是对的；
- hand 内部的默认通信应该仍是 **self-attention + graph bias**；
- 若要加 cross-attention，应该加在 **dynamic joint stream ↔ static embodiment stream**，以及 **hand ↔ object memory** 两类跨模态链路，而不是替代整套 hand graph。

### 三、关于 object-side：**T(R,O) 最值得借的是“显式 hand-object relation”，不是 patch graph 本体**

从 `tro_graph.py` / `denoiser.py` 可见：
- 物侧是多 patch token；
- 手侧是 link nodes；
- OR / RR 两类边都带显式 SE(3) 关系；
- 整套系统是为 diffusion grasp synthesis 设计的，在线闭环 policy 太重。

**结构含义**：
- 我们不该把 object side 做成完整 patch graph；
- 但也不该退化成完全无结构的纯 `[CLS]`；
- 更合理的折中是：
  - 让 `z_f` 聚合出 **少量 object memory slots**；
  - 这些 slots 作为 hand 读取的 object-side memory；
  - 若以后要加显式 hand-object relation，优先加在 distal joints / fingertips 到 memory 的交互上，而不是引入大量 object patches。

### 四、关于 local / global supervision：**文献支持“local 学 interaction，global 学 object-level state”**

1. **HORA / RotateIt** 的 global latent 本质上学的是 object/extrinsics；
2. **AnyRotate** 明确说明 rich tactile 的价值在于检测局部接触不稳定、恢复 grasp；
3. **DexNDM** 强调 factorization 的价值在于把预测目标局部化，从而过滤与本局部无关的高维噪声。

**结构含义**：
- `z_f` 不应该都去回归同一个完整 object state；
- 更合理的是：
  - **local loss**：预测本指未来 contact / contact pose / force proxy / slip tendency
  - **global loss**：预测 privileged object extrinsics / object pose / angular velocity / coarse geometry code

### 五、关于 finger summary layer：**单手 MVP 不是硬必需，但多手型故事里价值明显上升**

1. **MAGCLA** 说明 finger-level cooperation 确实是可解释层；
2. **GET-Zero** 证明只用 joint layer 也能学，但它的泛化对象主要还是 LEAP family；
3. 如果未来要把 “不同 joint 数量 / 不同 finger layout” 对齐起来，显式 finger summary 会是很自然的中间语义层。

**结构含义**：
- 单手型 MVP：可先不上 finger summary，减少实现负担；
- 若第三轮目标是直接把 future multi-hand 接口也设计好：可以考虑保留一个 **可选的 finger pooling / summary hook**，但不一定第一版就启用复杂的双向 decode。

### 六、当前最像样的架构收缩版本

综合论文与代码证据，第三轮里最稳、也最像“能开始编码”的骨架更像：

1. **Per-finger temporal encoder（图外）**：小 TCN/GRU，从本指 history 得到 `z_f`
2. **Object-side memory**：由 `{z_f}` 聚合出 `K` 个小 memory slots（推荐先考虑 `K=2~4`）
3. **Dynamic joint tokens**：当前 joint states / previous actions / local obs
4. **Static embodiment tokens**：joint limits / rest pose / link geometry / graph metadata
5. **Cross-Attn 0**：dynamic joint tokens 查询 static embodiment tokens
6. **Hand graph encoder**：self-attention + graph bias（SPD / parent-child / edge bias）
7. **Cross-Attn 1**：hand tokens 读取 object memory slots
8. **Action head**：joint-space output，且保持 `local 主路 + global/object residual`

也就是更接近：
`finger-local history inference -> compact object memory -> embodiment-aware joint graph -> joint action`

而不是：
`all signals -> one huge transformer -> action`

**小结**：补读论文和代码后，第三轮最值得优先拍板的，不再是“要不要 graph / 要不要 RMA”这种大方向，而是三个实现层级问题：
1. object memory 用 `1 / 2 / 4` 个 slots？
2. finger summary layer 第一版上不上？
3. cross-attention 精确落在 `dynamic↔static` 和 `hand↔object` 的哪一处？

**待确认**：你下一步更想先收敛哪一项：`object memory 数量`、`finger summary layer`，还是 `cross-attention 的具体拓扑`？

## 3.3 回到可编码层面的最小分歧

**用户反馈**：希望讨论语气更正常、直接，不要过度渲染。

**分析**：

在补读论文和代码之后，当前对架构最关键的判断可以收敛成下面三条：

1. **时序模块优先放图外**
   - 对当前 `20Hz+` 约束更稳；
   - 与 HORA / AnyRotate 的训练形态更一致；
   - 也更容易把 `per-finger latent` 和 `joint-level graph` 解耦。

2. **hand 内部默认仍用 self-attention + graph bias**
   - 这是 GET-Zero 最有价值的部分；
   - cross-attention 更适合放在跨模态交互，而不是取代 hand graph 本身。

3. **接下来最值得拍板的是 object-side memory 的最小形态**
   - 如果只有 1 个 global token，那么 hand->object 的 cross-attention 很容易退化成门控；
   - 如果一开始就上完整 patch graph，又明显超出第一版在线策略的复杂度预算；
   - 因此更现实的分歧是：`1 个 token` 还是 `少量 slots（2~4）`。

在这个前提下，当前最接近“可开始编码”的最小骨架是：
- 每指一个 history encoder，输出 `z_f`
- `{z_f}` 聚合成 object-side memory
- dynamic joint tokens 查询 static embodiment tokens
- hand graph 用 graph bias 做 joint-level message passing
- hand tokens 再读取 object-side memory
- action 仍然输出 joint-space，并保留 local 主路 + global 残差

**小结**：大的方向已经基本清楚，剩下最影响实现的是 `object memory` 以及它对应的 `cross-attention` 形态。

**待确认**：下一步是先把 object memory 定成单 token 还是少量 slots？

## 3.4 交叉注意力（Cross-Attention）在当前架构里的作用

**用户反馈**：希望后续讨论采用更清楚、讲原理、分点说明的表达方式，并继续聚焦 cross-attention 的具体作用与放置位置。

**分析**：

### 一、交叉注意力在本问题里的定义

在当前讨论中，我们有两类不同来源的序列：

1. **Hand-side tokens**
   - joint-level dynamic tokens：来自当前关节状态、上一时刻动作、局部观测
   - 或进一步得到的 finger/joint summary tokens

2. **Condition-side tokens**
   - static embodiment tokens：URDF、joint limit、rest pose、link geometry、graph meta-data
   - object-side memory tokens：由 per-finger history encoder 输出的 `{z_f}` 聚合而来

如果记 hand-side 序列为 $X \in \mathbb{R}^{L_x 	imes d}$，condition-side 序列为 $Y \in \mathbb{R}^{L_y 	imes d}$，
则交叉注意力可以写成：

$$
Q = X W_Q, \quad K = Y W_K, \quad V = Y W_V
$$

$$
\mathrm{CrossAttn}(X, Y)=\mathrm{softmax}\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

其中输出长度由 $X$ 决定，也就是：
- **谁提供 $Q$，谁就决定“我要更新谁”**；
- **谁提供 $K,V$，谁就决定“我能从哪里取信息”**。

### 二、为什么这里不能把所有交互都写成 self-attention

如果把所有 token（joint、embodiment、object memory）直接拼成一个大序列做 self-attention，当然在数学上可行；但从结构上会有三个问题：

1. **语义混合过早**
   - joint state 是动态控制量；
   - embodiment token 是静态结构先验；
   - object memory 是交互记忆。
   - 三者统计性质不同，直接混在一起会增加 disentangle 的难度。

2. **可解释性变差**
   - 很难明确回答“当前 joint token 到底主要在查询 hand structure，还是在查询 object state”。

3. **复杂度与调试成本更高**
   - 对第一版可编码架构不利。

因此，在当前问题里，把 cross-attention 作为**跨模态信息注入模块**，而把 self-attention + graph bias 作为**hand 内部结构建模模块**，会更清楚。

### 三、当前最自然的两处 cross-attention

#### 1. Dynamic joint stream × static embodiment stream

这里的目标是：
> 当前 joint 的数值状态，应该如何在“这只手的结构”里被解释？

设：
- dynamic joint tokens 为 $X_{dyn}$
- static embodiment tokens 为 $Y_{emb}$

则可写为：

$$
H_{emb} = \mathrm{CrossAttn}(X_{dyn}, Y_{emb})
$$

其含义是：
- $Q$ 来自 dynamic joint tokens，表示“当前关节状态提出的问题”；
- $K,V$ 来自 embodiment tokens，表示“这只手的结构知识库”；
- 输出仍是 joint-length 序列，因此后续 action head 仍能保持 joint-space 对齐。

**优点**：
- 符合 hand-generalization 的核心需求；
- 比简单 concat 更容易解释；
- 不破坏 joint-level tokenization。

#### 2. Hand-side tokens × object-side memory

这里的目标是：
> 当前 hand token 需要从 object interaction memory 中读取什么上下文，来修正控制？

设：
- hand tokens 为 $X_{hand}$
- object memory 为 $Y_{obj}$

则可写为：

$$
H_{obj} = \mathrm{CrossAttn}(X_{hand}, Y_{obj})
$$

其含义是：
- hand token 作为 query；
- object memory 作为 key/value；
- 输出长度依然与 hand token 一致，所以适合直接接 action head 或 residual head。

### 四、哪些地方更适合 self-attention，而不是 cross-attention

#### 1. hand 内部 joint-to-joint 通信

这部分更适合：

$$
\mathrm{SelfAttn}(X_{hand}) + \mathrm{GraphBias}
$$

原因是：
- joint 之间原本就处于同一结构系统内；
- GET-Zero 已经证明 graph bias 对这种通信是有效的；
- 这里的核心问题不是“从外部记忆检索信息”，而是“在关节图内部传播信息”。

#### 2. `{z_f}` 到 object memory 的聚合

这里不一定必须用 cross-attention。

如果 `{z_f}` 只是聚合成 1 个 global token，那么：
- mean pooling
- attention pooling
- small DeepSets aggregator

都可能足够。

只有当 object side 明确采用多个 memory slots 时，才更有必要用：
- learned slots 作为 query
- `{z_f}` 作为 key/value

即：

$$
M_{obj} = \mathrm{CrossAttn}(X_{slot}, \{z_f\})
$$

此时 cross-attention 的意义才比较明确：
- 每个 slot 主动从所有 finger local latent 中“分工读取”不同信息。

### 五、从当前证据看，第一版最合理的 cross-attention 拓扑

如果目标是：
- joint-space action
- 20Hz+
- 单手 MVP 先跑通
- 未来保留多手型扩展接口

那么当前最合理的顺序是：

1. **per-finger history encoder（图外）**
   $$
   z_f = \phi_f(h_f)
   $$

2. **`{z_f}` -> object memory**
   - 若 memory 很小，可先用 pooling；
   - 若用 2~4 个 slots，可用 slot-to-finger cross-attention。

3. **dynamic joint tokens 查询 static embodiment tokens**
   $$
   X_{emb} = \mathrm{CrossAttn}(X_{dyn}, Y_{emb})
   $$

4. **hand graph encoder 做内部通信**
   $$
   X_{hand} = \mathrm{SelfAttn}(X_{emb}) + \mathrm{GraphBias}
   $$

5. **hand tokens 查询 object memory**
   $$
   X_{out} = \mathrm{CrossAttn}(X_{hand}, Y_{obj})
   $$

6. **joint-space action head**
   $$
   a_j = \pi_{local}(x_j, z_{f(j)}, e_j) + \Delta \pi_{global}(x_j, Y_{obj})
   $$

### 六、一个关键判断

当前最影响 cross-attention 形态的，不是“要不要用 cross-attention”，而是：

> **object-side memory 到底是 1 个 token，还是多个 slots？**

因为：
- 如果只有 **1 个 token**，hand->object 的 cross-attention 很容易退化成一种全局 gating；
- 如果有 **2~4 个 slots**，hand token 才真正有“按需检索不同 object context”的空间。

**小结**：
- hand 内部：优先 `self-attention + graph bias`
- hand 与 embodiment / object memory：优先 `cross-attention`
- object-side 是否采用多个 memory slots，将直接决定 cross-attention 是否只是门控，还是具备真实的信息路由能力。

**待确认**：如果我们现在专门讨论 cross-attention 的具体拓扑，那么下一步最关键的是先决定：`object-side memory` 用单 token，还是用少量 slots？

## 3.5 Graph Transformer 还是 MPNN

**用户反馈**：在继续讨论 object-side memory 之前，先判断 hand-side 主体网络用 Graph Transformer 还是 MPNN 更合适。

**分析**：

### 一、先明确两者的核心差别

#### 1. MPNN（Message Passing Neural Network）

MPNN 的基本形式可以写成：

$$
m_i^{(l)} = \sum_{j \in \mathcal{N}(i)} \phi_e\big(h_i^{(l)}, h_j^{(l)}, e_{ij}\big)
$$

$$
h_i^{(l+1)} = \phi_v\big(h_i^{(l)}, m_i^{(l)}\big)
$$

其中：
- $h_i^{(l)}$ 是第 $l$ 层节点 $i$ 的特征；
- $e_{ij}$ 是边特征；
- $\mathcal{N}(i)$ 是节点 $i$ 的邻居集合。

**核心特点**：
- 只在图的局部邻域传递信息；
- 结构稀疏，计算简单；
- 更接近“沿着图逐层扩散信息”。

#### 2. Graph Transformer

Graph Transformer 的基本注意力形式可以写成：

$$
A_{ij} = 
rac{Q_i K_j^T}{\sqrt{d_k}} + b_{ij}
$$

$$
\mathrm{Attn}(i) = \sum_j \mathrm{softmax}(A_{ij}) V_j
$$

其中 $b_{ij}$ 是图结构偏置，例如：
- shortest path distance (SPD)
- parent-child bias
- edge-type bias

**核心特点**：
- 本质上仍允许全局通信；
- 图结构不是通过“限制只能连邻居”实现，而是通过 bias 改变注意力分布；
- 更适合在局部结构与全局依赖之间做软权衡。

---

### 二、放到我们当前问题里，二者分别有什么优劣

#### 1. 如果用 MPNN

**优点**：
1. **更轻量**
   - 对第一版 20Hz+ 在线策略更友好；
   - 更容易控制显存和计算量。

2. **更符合稀疏物理直觉**
   - joint 只与 parent/child 或 finger 邻域通信；
   - 如果你非常强调“物理邻接优先”，MPNN 很自然。

3. **工程实现更直接**
   - 尤其当 hand graph 很固定时，调试成本低。

**缺点**：
1. **远程依赖传播慢**
   - 跨手指协调要经过多层 message passing；
   - 如果 object interaction 需要快速影响多个 finger，MPNN 可能需要更多层。

2. **跨模态整合不自然**
   - embodiment tokens、object memory tokens、joint tokens 放在同一框架下时，不如 attention 统一；
   - cross-attention 风格接口不如 transformer 体系自然。

3. **和 GET-Zero 的主线不完全对齐**
   - GET-Zero 的关键证据是 joint token + graph-biased attention，而不是经典 MPNN。

#### 2. 如果用 Graph Transformer

**优点**：
1. **更适合当前已收敛的 joint-level token 设计**
   - GET-Zero 已经给出了现成证据：joint tokens + graph bias 在 embodiment-aware control 上有效。

2. **更适合处理“局部 + 全局”并存的问题**
   - hand 内部是 graph-biased self-attention；
   - hand 与 embodiment / object memory 可以无缝接 cross-attention；
   - 架构更统一。

3. **更适合未来多手型扩展**
   - token 数变化、手指结构变化、memory slot 数变化都更容易兼容；
   - 对未来增加 finger summary layer 也更自然。

**缺点**：
1. **实现和调试更复杂**
   - attention bias、padding、不同 token 类型处理都会增加工程负担。

2. **如果不控制好，容易过重**
   - 尤其当你把时序、embodiment、object memory 全都塞进一个大 attention 框架时。

3. **第一版可能“用力过猛”**
   - 如果目标只是单手 MVP，Graph Transformer 可能比 MPNN 更难训、更难排错。

---

### 三、和当前 5 个架构问题的耦合关系

#### 1. 对 object memory 的影响

- **MPNN** 下，object memory 更像额外节点或外部条件，通常会更倾向少量节点甚至单一 global token；
- **Graph Transformer** 下，多个 memory slots 更自然，因为 token-based cross-attention 更统一。

#### 2. 对 cross-attention 的影响

- **MPNN**：cross-attention 更像外挂模块，主体 graph 和跨模态模块是分离的；
- **Graph Transformer**：self-attention / cross-attention 可以放在同一套 token 框架下，设计更整齐。

#### 3. 对 finger summary layer 的影响

- **MPNN**：如果以后上 finger summary，可能需要额外设计 pooling + broadcast 机制；
- **Graph Transformer**：更容易把 finger summary 看成额外 token 层。

#### 4. 对时序模块的影响

无论选哪一个，我仍然认为：
- 第一版 history encoder 更适合放图外；
- 不建议把长时序直接并进 hand graph 主体。

这一点并不会因为选 MPNN 还是 Graph Transformer 而改变。

---

### 四、结合当前项目约束的判断

如果我们把目标拆成两个层次：

#### 目标 A：第一版尽快可跑、单手 MVP、20Hz 优先

那么 **MPNN 更有吸引力**，因为：
- 更简单；
- 更稀疏；
- 更容易在 object memory 还是不太确定时先搭一个可运行版本。

#### 目标 B：第一版就尽量贴近最终 paper 主架构，未来要兼容多手型、cross-attn、memory slots

那么 **Graph Transformer 更合适**，因为：
- 与 GET-Zero 的证据链更一致；
- 更适合 joint token + embodiment tokens + object memory 的统一表达；
- 更方便后续加 cross-attention 与 finger summary。

---

### 五、当前更实际的建议

如果只考虑当前已经收敛的内容：
- joint-level token
- dynamic state × static embodiment cross-attention
- object-side memory
- local 主路 + global 残差

那么我认为：

> **主干更适合选 Graph Transformer，而不是纯 MPNN。**

原因不是因为它“更高级”，而是因为它和我们当前已经接受的其他设计更一致。

如果现在改成 MPNN，会出现一个问题：
- hand 主干是 MPNN；
- embodiment 交互要单独做 cross-attention；
- object memory 交互也要单独做 cross-attention；
- 最后整体会变成“局部图网络 + 两个外挂注意力模块”的混合系统。

这不是不能做，但结构会更碎。

相反，Graph Transformer 可以更统一地表达：
- hand 内部：graph-biased self-attention
- hand↔embodiment：cross-attention
- hand↔object memory：cross-attention

唯一需要控制的是**不要把时序也塞进去**，否则会过重。

---

### 六、一个折中思路

如果你担心 Graph Transformer 太重，但又不想失去它的统一表达能力，可以考虑：

1. **只在 hand graph 主干上用很小的 Graph Transformer**
   - 层数少；
   - token 数只保留 joint tokens；
   - graph bias 用 GET-Zero 风格。

2. **history encoder 放图外**
   - 每指小 TCN/GRU 先得到 `z_f`。

3. **object-side memory 保持很小**
   - 先从 1 或 2 个 slots 开始。

这会把复杂度控制在一个比较合理的范围内。

**小结**：
- 若只求最简 MVP：MPNN 有优势；
- 若考虑当前已收敛的整体架构一致性：Graph Transformer 更合适；
- 在当前讨论上下文里，我更倾向于 **小型 Graph Transformer**，而不是纯 MPNN。

**待确认**：你是更想要“结构统一、贴近最终 paper 主架构”，还是“先用更朴素的 MPNN 快速做一个最小原型”？

## 3.6 若追求双向泛化，GET-Zero 还是 T(R,O)-Grasp 更适合作为 backbone**用户反馈**：问题不是抽象地比较 Graph Transformer 和 MPNN，而是更具体地比较：如果要追求 hand × object 双向泛化，应该更接近 GET-Zero 还是 T(R,O)-Grasp 的 backbone 设计。**分析**：### 一、先明确两个“backbone”分别在解决什么问题#### 1. GET-Zero 的 backbone 在解决什么GET-Zero 的核心目标是：> **embodiment-aware control**它最关键的结构特征是：- joint-level tokenization- graph-biased self-attention- robot-only embodiment graph- online policy network它擅长解决的是：- 不同 joint 数量 / 结构变化下的控制表示；- 把 hand morphology 编进策略主干；- 在在线控制频率下运行。#### 2. T(R,O)-Grasp 的 backbone 在解决什么T(R,O)-Grasp 的核心目标是：> **cross-embodiment dexterous grasp synthesis**它最关键的结构特征是：- object patch nodes + robot link nodes- OR / RR edges 带显式相对 SE(3)- graph diffusion- 更偏生成式 grasp synthesis，而不是在线闭环控制它擅长解决的是：- 显式表示 hand-object spatial relation；- 在不同手和不同物体之间建立统一几何关系；- 生成 grasp，而不是逐时刻输出 joint action。---### 二、如果目标是“双向泛化的 in-hand manipulation”，主干最需要优先满足什么在我们的问题里，主 backbone 至少要同时满足三件事：1. **能做在线 joint-space control**   - 不是只生成 grasp；   - 而是每一步都输出 joint action。2. **能编码 hand embodiment 差异**   - 否则 hand 泛化很难成立。3. **能接入 object-side interaction representation**   - 否则 object 泛化会退化成弱版本的全局 latent。这三点里，前两点是“主干硬约束”，第三点更像“需要外挂/扩展模块”。---### 三、为什么我认为 GET-Zero 更适合作为主 backbone#### 1. 它天然就是 policy backboneGET-Zero 本身就是为了控制策略设计的：- joint token 输入；- per-joint head 输出；- graph bias 直接作用在在线策略主干里。这和我们的需求高度一致。#### 2. 它天然支持 embodiment-aware tokenization如果目标包含 hand 泛化，那么 backbone 必须先能回答：> 这只手的 joint / link 结构差异，如何进入控制网络？GET-Zero 对这个问题已经有比较成熟的答案：- token 粒度是 joint；- bias 编码结构；- 输出仍保持 per-joint 对齐。#### 3. 它更符合 20Hz+ 的工程约束T(R,O)-Grasp 的主干虽然表达力强，但它是为 graph diffusion grasp synthesis 设计的。即使它的训练和推理效率已经优于 D(R,O)，它的结构重点仍然是：- patch graph- relation graph- diffusion denoising这套东西更像“生成 grasp”的 backbone，而不是“每个 control step 都要跑”的 backbone。---### 四、为什么 T(R,O)-Grasp 不能直接作为主 backbone这里不是说它不好，而是说它的长板和我们的主需求错位。#### 1. 它更偏 grasp synthesis，不是 online manipulation policyT(R,O)-Grasp 输出的是 grasp-related spatial transformation，而我们的任务输出的是：$$a_t \in \mathbb{R}^{	ext{DoF}}$$也就是 joint-space action。这意味着：- 如果直接拿它做 backbone，后面还要额外接一层“从 graph state 到在线控制”的体系；- 主干和任务目标之间仍隔着一层很厚的桥。#### 2. 它的 object 表达太重，不适合第一版在线策略T(R,O)-Grasp 的 object side 本质上依赖 patch graph。这对 grasp synthesis 很合理，因为它强调空间几何精度。但在当前问题里，我们更关心：- 持续控制；- 20Hz+；- 单手 MVP 能先起来。完整 patch graph 会明显拉高状态维护、计算和调试复杂度。#### 3. 它对 hand-generalization 的主线支持不如 GET-Zero 直接T(R,O)-Grasp 虽然做 cross-embodiment grasp，但它的“控制主干”不是 joint-level policy backbone。而我们当前更需要一个能够把 hand embodiment 差异稳稳放进在线策略主干的结构。---### 五、但为什么又必须借 T(R,O)-Grasp因为 GET-Zero 也有明显短板：> **它没有 object，也没有 hand-object relation。**如果完全沿 GET-Zero 主干走而不借 T(R,O)-Grasp，会出现一个问题：- hand 泛化主线很强；- 但 object side 可能会重新退化成“一个弱的全局 latent”；- 最后很难真正支撑“hand × object 双向泛化”的故事。所以 T(R,O)-Grasp 最值得借的不是整个 backbone，而是两点：1. **显式 hand-object relation 的思想**   - 即 hand side 与 object side 的交互，不应该只是简单 concat。2. **object / robot 分类型节点的建模意识**   - 即 object side 和 hand side 可以是异构结构，而不是完全平铺在一个序列里。---### 六、因此更合理的组合方式如果把“主 backbone”和“结构借鉴来源”分开看，我更推荐下面这个组合：#### 主 backbone：GET-Zero 风格- joint-level tokens- graph-biased self-attention- per-joint action head- embodiment-aware static tokens#### object-side / interaction 机制：借 T(R,O)-Grasp- 不直接用 patch graph- 但保留“hand-object relation 应该显式建模”的思想- 用更轻量的 object memory / interaction slots 代替 patch nodes- 用 cross-attention 或 relation-conditioned bias 建立 hand ↔ object interaction也就是说：> **主干用 GET-Zero 的语法，object-side 用 T(R,O)-Grasp 的思想。**---### 七、如果真的要二选一如果必须回答“更像 GET-Zero 还是更像 T(R,O)-Grasp”，我的答案是：> **更像 GET-Zero。**原因是：- 它更像 online policy backbone；- 它更适合 joint-space action；- 它更能承担 hand-generalization 主线；- 它更符合我们当前的 20Hz+ 工程约束。而 T(R,O)-Grasp 应该更多作为：- object-side relation 建模的灵感来源；- 不是 policy 主干本身。**小结**：- 若目标是 hand × object 双向泛化的 in-hand manipulation，主 backbone 更适合沿 GET-Zero；- 但必须显式补上 GET-Zero 所没有的 object-side interaction，这部分最值得借 T(R,O)-Grasp 的设计思想。**待确认**：你是否接受这个方向——**主干以 GET-Zero 风格为主，object-side 机制借 T(R,O)-Grasp，但不直接使用完整 patch graph**？